# Regular Expressions

**Seong Cho** · ACME Data Science Essentials, Lab 3

**Lab objective:** use regular expressions to match, search, and manipulate text, and apply them to a data cleaning problem.

<sub>Lab specification from the *Data Science Essentials* lab manual of [Foundations of Applied Mathematics](https://foundations-of-applied-mathematics.github.io/) (BYU ACME), © Dr. J. Humpherys, used under [CC BY 3.0 US](http://creativecommons.org/licenses/by/3.0/us/). Problem statements are quoted from my lab template and the spec; the notes are condensed from the spec. Solutions are my own.</sub>

In [1]:
# regular_expressions.py
# """Volume 3: Regular Expressions.
# Seong-Eun Cho
# Volume3
# 9/10/18
# """
import re

A **regular expression** (regex) is a string written in a special syntax that specifies a pattern; strings that follow the pattern **match** it. One regex can match a large set of strings, such as all valid email addresses.

> **Achtung!** Regex syntax is broadly standard but varies slightly between programs and languages. The Python syntax here is close enough to carry over to Vim or other editors, with small differences.

## The `re` module

`re.compile()` turns a pattern string into a reusable pattern object. Its `search()` method returns a match object if any part of the target string matches (and `None` otherwise), while `match()` only matches at the *beginning* of the string. Module-level shortcuts such as `re.search(pattern, s)` recompile the pattern on every call, so compiling once is good practice.

In [2]:
pattern = re.compile("cat")
print(bool(pattern.match("catfish")), bool(pattern.match("fishcat")), bool(pattern.search("fishcat")))

True False True


## Problem 1: A first pattern

> **Problem 1.** Compile and return a regular expression pattern object with the pattern string "python".

In [3]:
# Problem 1
def prob1():
    """Compile and return a regular expression pattern object with the
    pattern string "python".

    Returns:
        (_sre.SRE_Pattern): a compiled regular expression pattern object.
    """
    return re.compile(r"python")

In [4]:
# Test prob1
print("Testing prob1")
print(prob1())

Testing prob1
re.compile('python')


## Literal characters and metacharacters

The **metacharacters** `. ^ $ * + ? { } [ ] \ | ( )` have special meanings. To match them literally, use a raw string (`r"..."`, so backslashes stay backslashes) and escape each metacharacter with a backslash: `"$3.99? Thanks."` is matched by `r"\$3\.99\? Thanks\."`.

## Problem 2: Escaping metacharacters

> **Problem 2.** Compile and return a regular expression pattern object that matches the string `"^{@}(?)[%]{.}(*)[_]{&}$"`.

In [5]:
# Problem 2
def prob2():
    """Compile and return a regular expression pattern object that matches
    the string "^{@}(?)[%]{.}(*)[_]{&}$".

    Returns:
        (_sre.SRE_Pattern): a compiled regular expression pattern object.
    """
    return re.compile(r"\^\{\@\}\(\?\)\[\%\]\{\.\}\(\*\)\[\_\]\{\&\}\$")

In [6]:
# Test prob2
print("\nTesting prob2")
print(prob2())
print("^{@}(?)[%]{.}(*)[_]{&}$")
print(r"^{@}(?)[%]\{.\}(*)[_]{&}$")    # [compat] raw string: "\{" is an invalid escape sequence (SyntaxWarning on Python 3.12+)
print(bool(re.match(prob2(), "^{@}(?)[%]{.}(*)[_]{&}$")))


Testing prob2
re.compile('\\^\\{\\@\\}\\(\\?\\)\\[\\%\\]\\{\\.\\}\\(\\*\\)\\[\\_\\]\\{\\&\\}\\$')
^{@}(?)[%]{.}(*)[_]{&}$
^{@}(?)[%]\{.\}(*)[_]{&}$
True


## Anchors, alternation, and groups

The line anchors `^` and `$` match the start and end of a line, so `'^x$'` matches only `'x'` while `'x'` matches anything containing an x. The pipe `|` is a logical OR, and parentheses `()` create a **group** that sets the order of operations: `"^one|two fish$"` means `^one` or `two fish$`, while `"^(one|two) fish$"` matches exactly `one fish` and `two fish`.

## Problem 3: Stores and suppliers

> **Problem 3.** Compile and return a regular expression pattern object that matches the following strings (and no other strings).
>
> | | | |
> |---|---|---|
> | Book store | Mattress store | Grocery store |
> | Book supplier | Mattress supplier | Grocery supplier |

In [7]:
# Problem 3
def prob3():
    """Compile and return a regular expression pattern object that matches
    the following strings (and no other strings).

        Book store          Mattress store          Grocery store
        Book supplier       Mattress supplier       Grocery supplier

    Returns:
        (_sre.SRE_Pattern): a compiled regular expression pattern object.
    """
    return re.compile(r"^(Book|Mattress|Grocery) (store|supplier)$")

In [8]:
# Test prob3
print("\nTesting prob3")
print(bool(re.search(prob3(), "Book store")))
print(bool(re.search(prob3(), "Mattress store")))
print(bool(re.search(prob3(), "Grocery store")))
print(bool(re.search(prob3(), "Book supplier")))
print(bool(re.search(prob3(), "Mattress supplier")))
print(bool(re.search(prob3(), "Grocery supplier")))


Testing prob3
True
True
True
True
True
True


## Character classes

Square brackets make a **character class**, which matches exactly *one* character from a set: `[abc]` matches `a`, `b` or `c` (but not `ab`). A leading `^` complements the set (`[^0-9]` is any non-digit) and `-` gives a range (`[a-z]`); a `-` at either end is a literal dash. A period `.` matches any character except a newline.

| Shortcut | Matches |
|---|---|
| `\b` | the empty string at the start or end of a word |
| `\s` / `\S` | any whitespace / non-whitespace character |
| `\d` / `\D` | any digit / non-digit |
| `\w` / `\W` | any alphanumeric character or `_` / anything else |

## Repetition

| Operator | Matches |
|---|---|
| `*` | 0 or more repetitions of the preceding expression |
| `+` | 1 or more repetitions |
| `?` | 0 or 1 |
| `{m,n}` | from `m` to `n` repetitions (`{,n}`, `{m,}`, `{k}` also work) |
| `*?`, `+?`, `??`, `{m,n}?` | non-greedy versions of the above |

Each operator applies to the expression just before it (a character, class, or group). Anchors matter with repetition: `r"a{3}"` finds `aaa` inside `aaaab`, whereas `r"^a{3}$"` matches only `aaa`.

## Problem 4: Python identifiers and parameters

> **Problem 4.** A valid Python identifier (a valid variable name) is any string starting with an alphabetic character or an underscore, followed by any (possibly empty) sequence of alphanumeric characters and underscores.
>
> A valid python parameter definition is defined as the concatenation of the following strings:
> - any valid python identifier
> - any number of spaces
> - (optional) an equals sign followed by any number of spaces and ending with one of the following: any real number, a single quote followed by any number of non-single-quote characters followed by a single quote, or any valid python identifier
>
> Define a function that compiles and returns a regular expression pattern object that matches any valid Python parameter definition.
>
> (Hint: Use the `\w` character class shortcut to keep your regular expression clean.)
>
> | Matches | Non-matches |
> |---|---|
> | `"Mouse"` `"compile"` `"_123456789"` `"__x__"` `"while"` | `"3rats"` `"err*r"` `"sq(x)"` `"sleep()"` `" x"` |
> | `"max=4.2"` `"string= ''"` `"num_guesses"` | `"300"` `"is_4=(value==4)"` `"pattern = r'^one\|two fish$'"` |

In [9]:
# Problem 4
def prob4():
    """Compile and return a regular expression pattern object that matches
    any valid Python identifier.

    Returns:
        (_sre.SRE_Pattern): a compiled regular expression pattern object.
    """
    return re.compile(r"^[a-zA-Z_][\w_]* *(\= *(\d+\.?\d*|\'[^\']*\'|[a-zA-Z_][\w_]*))?$")

In [10]:
# Test prob4
print("\nTesting prob4")
print("Matches:")
print(bool(re.match(prob4(), "Mouse")))
print(bool(re.match(prob4(), "compile")))
print(bool(re.match(prob4(), "_123456789")))
print(bool(re.match(prob4(), "__x__")))
print(bool(re.match(prob4(), "while")))
print("Non-matches:")
print(bool(re.match(prob4(), "3rats")))
print(bool(re.match(prob4(), "err*r")))
print(bool(re.match(prob4(), "sq(x)")))
print(bool(re.match(prob4(), "sleep()")))
print(bool(re.match(prob4(), " x")))
print("Matches:")
print(bool(re.match(prob4(), "max=4.2")))
print(bool(re.match(prob4(), "string= ''")))
print(bool(re.match(prob4(), "num_guesses")))
print("Non-matches:")
print(bool(re.match(prob4(), "300")))
print(bool(re.match(prob4(), "is_4=(value==4)")))
print(bool(re.match(prob4(), "pattern = r'^one|two fish$'")))


Testing prob4
Matches:
True
True
True
True
True
Non-matches:
False
False
False
False
False
Matches:
True
True
True
Non-matches:
False
False
False


## Manipulating text

Besides `search()` and `match()`, pattern objects have `fullmatch()` (match the whole string), `sub()` / `subn()` (substitute matches; `subn` also counts them), `split()`, `findall()`, and `finditer()`. In a substitution string, `\1`, `\2`, … refer back to the text matched by each group, e.g. `re.compile(r"\bcat(\w*)").sub(r"at\1clay", ...)` turns `catfish` into `atfishclay`.

The repetition operators are **greedy** (they match as much as possible): `r"<.*>"` swallows all of `"<abc> <def> <ghi>"`, while the non-greedy `r"<.*?>"` finds each `<...>` separately.

Flags passed to `re.compile()` customize matching: `re.DOTALL` (`.` also matches newlines), `re.IGNORECASE`, and `re.MULTILINE` (`^` and `$` also match at the start and end of each line).

## Problem 5: Missing colons

> **Problem 5.** Use regular expressions to place colons in the appropriate spots of the input string, representing Python code. You may assume that every possible colon is missing in the input string.

In [11]:
# Problem 5
def prob5(code):
    """Use regular expressions to place colons in the appropriate spots of the
    input string, representing Python code. You may assume that every possible
    colon is missing in the input string.

    Parameters:
        code (str): a string of Python code without any colons.

    Returns:
        (str): code, but with the colons inserted in the right places.
    """
    # Define the patterns for each class of keywords
    pattern1 = re.compile(r"(if|elif|for|while|with|def|class)(.+)")
    pattern2 = re.compile(r"(else|try|finally)")
    pattern3 = re.compile(r"(except.*)")

    # Substitue in the colons
    code = pattern1.sub(r"\1\2:", code)
    code = pattern2.sub(r"\1:", code)
    code = pattern3.sub(r"\1:", code)

    return code

In [12]:
# Test prob5
print("\nTesting prob5")
code = (
    "k, i, p = 999, 1, 0\n"
    "while k > i\n"
    "\ti *= 2\n"
    "\tp += 1\n"
    "\tif k != 999\n"
    "\t\tprint(\"k should not have changed\") \n"
    "\telse\n"
    "\t\tpass\n"
    "print(p)")

print(code)
print(prob5(code))


Testing prob5
k, i, p = 999, 1, 0
while k > i
	i *= 2
	p += 1
	if k != 999
		print("k should not have changed") 
	else
		pass
print(p)
k, i, p = 999, 1, 0
while k > i:
	i *= 2
	p += 1
	if k != 999:
		print("k should not have changed") 
	else:
		pass
print(p)


## Problem 6: Cleaning contact data

> **Problem 6.** Use regular expressions to parse the data in the given file and format it uniformly, writing birthdays as mm/dd/yyyy and phone numbers as (xxx)xxx-xxxx. Construct a dictionary where the key is the name of an individual and the value is another dictionary containing their information. Each of these inner dictionaries should have the keys "birthday", "email", and "phone". In the case of missing data, map the key to None.

In [13]:
# Problem 6
def prob6(filename="fake_contacts.txt"):
    """Use regular expressions to parse the data in the given file and format
    it uniformly, writing birthdays as mm/dd/yyyy and phone numbers as
    (xxx)xxx-xxxx. Construct a dictionary where the key is the name of an
    individual and the value is another dictionary containing their
    information. Each of these inner dictionaries should have the keys
    "birthday", "email", and "phone". In the case of missing data, map the key
    to None.

    Returns:
        (dict): a dictionary mapping names to a dictionary of personal info.
    """
    # Open the file and save it as a list
    with open(filename, 'r') as infile:
        contacts = infile.readlines()
    
    # Determine the regexes for each field
    reg_name = re.compile(r'^\b[a-zA-Z]+\b ([A-Z]\. )?\b[a-zA-Z]+\b')
    reg_bday = re.compile(r'\d{1,2}\/\d{1,2}\/\d{2,4}')
    reg_email = re.compile(r'\S+@\w+(\.[a-zA-Z]+)+')
    reg_phone = re.compile(r'(\d\-)?\(?(\d{3,4})\)?\-?(\d{3,4})\-(\d{3,4})')

    contact_dict = dict()
    # Iterate through the contacts and save each entry into a dictionary
    for contact in contacts:
        d = dict()
        name = reg_name.search(contact).group(0)

        if reg_bday.search(contact):
            bday_list = reg_bday.search(contact).group(0).split('/')
            if len(bday_list[0]) == 1:
                bday_list[0] = "0" + bday_list[0]
            if len(bday_list[1]) == 1:
                bday_list[1] = "0" + bday_list[1]
            if len(bday_list[2]) == 2:
                bday_list[2] = "20" + bday_list[2]
            d["birthday"] = '/'.join(bday_list)
        else:
            d["birthday"] = None
        if reg_email.search(contact):
            d["email"] = reg_email.search(contact).group(0)
        else:
            d["email"] = None
        if reg_phone.search(contact):
            d["phone"] = reg_phone.sub(r"(\2)\3-\4", reg_phone.search(contact).group(0))
        else:
            d["phone"] = None

        contact_dict[name] = d

    return contact_dict

In [14]:
# Test prob6
print("\nTesting prob6")
contacts = prob6()
# print(contacts)    # Changed: the full dictionary has about 2000 entries; print the first five instead
for name in list(contacts)[:5]:
    print(name, contacts[name])


Testing prob6
John Doe {'birthday': '01/01/2099', 'email': 'john_doe90@hopefullynotarealaddress.com', 'phone': '(123)456-7890'}
Jane Smith {'birthday': None, 'email': None, 'phone': '(222)111-3333'}
Graiden T. Moses {'birthday': '02/22/2058', 'email': 'vehicula_58@gravidaAliquamtincidunt.net', 'phone': '(287)610-0471'}
Brock S. Schneider {'birthday': '10/16/2069', 'email': 'sit@vulputateposuerevulputate.superfake.com', 'phone': '(483)928-1252'}
Mary H. Hatfield {'birthday': '10/16/2035', 'email': 'nisi.nibh.lacinia@lectus.fake.com', 'phone': '(430)750-4067'}
